In [13]:
import pandas as pd

ama = pd.read_excel(r"amocrm_export_leads_2026-02-13.xlsx")
onec = pd.read_excel(r"Ю-АР-СИ.XLSX")
onec = onec[onec['Ответственный'] == 'Яшихин Алексей Валерьевич']

In [14]:
ama['Дата создания AMA'] = pd.to_datetime(ama['Дата создания'], dayfirst=True)
onec['Дата закрытия 1C'] = pd.to_datetime(onec['Дата выезда'], dayfirst=True)
onec['Дата создания 1C'] = pd.to_datetime(onec['Дата'], dayfirst=True)

C:\Users\a.vorona\AppData\Local\Temp\ipykernel_16132\3574478231.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ama['Дата создания AMA'] = pd.to_datetime(ama['Дата создания'], dayfirst=True)


ЕСли заказ создан в 1С, а в AMA на следующий день (24ч)

In [15]:
merged = ama.merge(onec, how='cross')
merged['weekday'] = merged['Дата создания AMA'].dt.weekday
merged['Начало 1С - Начало AMA (ч)'] = round((merged['Дата создания 1C'] - merged['Дата создания AMA']).dt.total_seconds() / 3600, 2)
merged['Начало AMA - Начало 1C (ч)'] = round((merged['Дата создания AMA'] - merged['Дата создания 1C']).dt.total_seconds() / 3600, 2)

result = merged[
    ((merged['weekday'] <= 4) & (merged['Начало 1С - Начало AMA (ч)'] <= 72) & (merged['Начало 1С - Начало AMA (ч)'] >= 0)) | 
    ((merged['weekday'] == 5) & (merged['Начало 1С - Начало AMA (ч)'] <= 120) & (merged['Начало 1С - Начало AMA (ч)'] >= 0)) |
    ((merged['weekday'] <= 5) & (merged['Начало AMA - Начало 1C (ч)'] <= 24) & (merged['Начало AMA - Начало 1C (ч)'] >= 0))

]


result['Конец 1C - Начало 1C (ч)'] = round((result['Дата закрытия 1C'] - result['Дата создания 1C']).dt.total_seconds() / 3600, 2)

C:\Users\a.vorona\AppData\Local\Temp\ipykernel_16132\1874363648.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged = ama.merge(onec, how='cross')
C:\Users\a.vorona\AppData\Local\Temp\ipykernel_16132\1874363648.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged['weekday'] = merged['Дата создания AMA'].dt.weekday
C:\Users\a.vorona\AppData\Local\Temp\ipykernel_16132\1874363648.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor pe

In [16]:
result.loc[:,['Номер', 'Дата создания AMA', 'Дата создания 1C', 'Начало 1С - Начало AMA (ч)', 'Начало AMA - Начало 1C (ч)', 'Конец 1C - Начало 1C (ч)', 'Лот_y']]

,Номер,Дата создания AMA,Дата создания 1C,Начало 1С - Начало AMA (ч),Начало AMA - Начало 1C (ч),Конец 1C - Начало 1C (ч),Лот_y
71,БТС00002596,2026-02-10 22:49:36,2026-02-10 11:55:19,-10.90,10.90,NaN,B 01104
72,БТС00002601,2026-02-10 22:49:36,2026-02-10 12:07:34,-10.70,10.70,NaN,B 01610
73,БТС00002755,2026-02-10 22:49:36,2026-02-11 15:32:55,16.72,-16.72,NaN,E 04743
74,БТС00002758,2026-02-10 22:49:36,2026-02-11 15:42:25,16.88,-16.88,NaN,E 04740
143,БТС00002117,2026-02-05 10:01:56,2026-02-04 11:59:36,-22.04,22.04,-11.99,B 02105
...,...,...,...,...,...,...,...
4131,БТС00014103,2025-09-15 17:03:47,2025-09-17 12:07:21,43.06,-43.06,131.88,B 02106
4202,БТС00012833,2025-08-29 09:53:31,2025-08-29 11:19:35,1.43,-1.43,-11.33,B 02106
4276,БТС00012654,2025-08-27 10:47:05,2025-08-27 11:13:43,0.44,-0.44,444.77,B 01101
4277,БТС00012833,2025-08-27 10:47:05,2025-08-29 11:19:35,48.54,-48.54,-11.33,B 02106


In [17]:
result = result.loc[result['Конец 1C - Начало 1C (ч)'] >= 0,
           ['Номер', 'Дата создания AMA', 'Дата создания 1C', 
            'Дата закрытия 1C', 'Начало 1С - Начало AMA (ч)', 'Начало AMA - Начало 1C (ч)', 'Конец 1C - Начало 1C (ч)', 'VIN_Х. (ПР)', 'Лот_y']]

In [18]:
result

,Номер,Дата создания AMA,Дата создания 1C,Дата закрытия 1C,Начало 1С - Начало AMA (ч),Начало AMA - Начало 1C (ч),Конец 1C - Начало 1C (ч),VIN_Х. (ПР),Лот_y
288,БТС00001770,2026-01-30 13:15:52,2026-01-30 16:11:49,2026-02-04,2.93,-2.93,103.80,E 04740,B 01562
363,БТС00001770,2026-01-30 12:07:12,2026-01-30 16:11:49,2026-02-04,4.08,-4.08,103.80,B 01361,B 01562
438,БТС00001770,2026-01-29 16:43:32,2026-01-30 16:11:49,2026-02-04,23.47,-23.47,103.80,B 01562,B 01562
513,БТС00001770,2026-01-29 16:42:59,2026-01-30 16:11:49,2026-02-04,23.48,-23.48,103.80,E 04740,B 01562
588,БТС00001770,2026-01-29 16:42:22,2026-01-30 16:11:49,2026-02-04,23.49,-23.49,103.80,B 01558,B 01562
...,...,...,...,...,...,...,...,...,...
4128,БТС00013979,2025-09-15 17:03:47,2025-09-16 10:08:58,2025-09-24,17.09,-17.09,181.85,B 02105,B 01104
4130,БТС00014102,2025-09-15 17:03:47,2025-09-17 12:05:41,2025-09-24,43.03,-43.03,155.91,B 02105,B 02105
4131,БТС00014103,2025-09-15 17:03:47,2025-09-17 12:07:21,2025-09-23,43.06,-43.06,131.88,B 02105,B 02106
4276,БТС00012654,2025-08-27 10:47:05,2025-08-27 11:13:43,2025-09-15,0.44,-0.44,444.77,B 01101,B 01101


In [19]:
df = result[result['Лот_y'] == result['VIN_Х. (ПР)']]
df_all = df.sort_values(by='Дата создания AMA', ascending=True)
display(df_all)
df_all.shape[0]

,Номер,Дата создания AMA,Дата создания 1C,Дата закрытия 1C,Начало 1С - Начало AMA (ч),Начало AMA - Начало 1C (ч),Конец 1C - Начало 1C (ч),VIN_Х. (ПР),Лот_y
4350,БТС00012300,2025-08-21 14:32:35,2025-08-21 17:48:25,2025-08-25,3.26,-3.26,78.19,B 02106,B 02106
4276,БТС00012654,2025-08-27 10:47:05,2025-08-27 11:13:43,2025-09-15,0.44,-0.44,444.77,B 01101,B 01101
4130,БТС00014102,2025-09-15 17:03:47,2025-09-17 12:05:41,2025-09-24,43.03,-43.03,155.91,B 02105,B 02105
4056,БТС00014103,2025-09-15 17:04:23,2025-09-17 12:07:21,2025-09-23,43.05,-43.05,131.88,B 02106,B 02106
3978,БТС00013979,2025-09-15 17:04:49,2025-09-16 10:08:58,2025-09-24,17.07,-17.07,181.85,B 01104,B 01104
3907,БТС00014946,2025-09-29 15:30:09,2025-09-29 15:21:36,2025-10-08,-0.14,0.14,200.64,B 01104,B 01104
3759,БТС00015977,2025-10-13 08:21:52,2025-10-13 08:54:30,2025-10-21,0.54,-0.54,183.09,B 02106,B 02106
3686,БТС00016503,2025-10-20 12:54:14,2025-10-20 11:23:04,2025-10-21,-1.52,1.52,12.62,B 02105,B 02105
3610,БТС00016496,2025-10-20 12:54:19,2025-10-20 11:07:48,2025-10-21,-1.78,1.78,12.87,B 01104,B 01104
3537,БТС00017023,2025-10-27 16:28:47,2025-10-27 10:17:12,2025-10-29,-6.19,6.19,37.71,B 02106,B 02106


34

In [ ]:
df_all.to_excel('result_new.xlsx')